# Revisión humana del audit de SMB

## Goal

Revisar de forma guiada y reproducible la muestra visual fijada de SMB y los candidatos a duplicado **sin ejecutar superresolución ni métricas**.

Este notebook carga la revisión inmutable configurada en `data/sources/smb.yaml`, conserva el CSV existente y guarda progresivamente las decisiones en `data/audits/smb-review-v1.csv`. No ejecuta ni debe ejecutar `prepare-review`.

Puedes cerrarlo y reabrirlo: las decisiones ya guardadas se conservan.

## Setup

### 1. Abrir el entorno

Inicia JupyterLab desde la raíz de `proyecto/`:

```bash
uv run --group notebooks jupyter lab
```

El acceso usa el login local de Hugging Face o `HF_TOKEN`; el token nunca se imprime ni se escribe en el notebook.

In [ ]:
from pathlib import Path

from score_super_resolution.smb_review_ui import SMBReviewSession

review = SMBReviewSession(Path.cwd())
review.summary()

## Steps

### 2. Confirmar la política general

La política propuesta:

- agrupa páginas retirando únicamente el sufijo `_pN` del identificador de partitura;
- confirma la licencia del dataset CC BY-NC 4.0 y el acceso autenticado;
- registra como no disponible la procedencia exacta por imagen;
- permite reproducción y redistribución no comercial con atribución;
- usa la auditoría automática para las páginas fuera de la muestra;
- conserva todos los elementos y añade las cuatro excepciones técnicas a la cola visual.

Introduce tu nombre, marca la confirmación y pulsa **Aplicar política**. Las filas ya revisadas no se sobrescriben.

In [ ]:
review.policy_widget()

### 3. Inspeccionar las páginas por lotes

No tienes que clasificar 67 páginas una a una. La muestra se presenta en **cinco lotes de hasta 16 páginas**. En cada lote:

1. el cuadro **IDs** contiene todos los identificadores en texto, uno por línea, para poder copiarlos;
2. pulsa **Mostrar lote** y recorre visualmente todas las miniaturas;
3. debajo de las imágenes aparece una cuadrícula 4×4 de checkboxes en el mismo orden: marca directamente las páginas que quieras ampliar;
4. confirma la inspección y pulsa **Aprobar lote**.

Las exclusiones se guardan inmediatamente en el CSV como una cola persistente de revisión individual. Si vuelves al lote o reinicias el kernel, sus checkboxes aparecerán marcados.

La aprobación por lote registra `acceptable` y `suitable` para las páginas no marcadas. Este es el resultado esperado para un dataset curado: la finalidad de la muestra es detectar excepciones materiales, no buscar defectos microscópicos.

#### Qué no debe marcarse como anomalía

- **Fondo amarillo o de otro tono:** es variación válida del soporte. Déjalo como `acceptable/suitable` si la tinta conserva buena separación del fondo.
- **Inclinación mínima:** no la marques. Usa `skewed` solo si se aprecia a vista de página completa y el desplazamiento de los pentagramas de un extremo al otro se aproxima al menos a un espacio de pentagrama o exigiría corregir la geometría.
- **Pocos pentagramas y mucho blanco:** es baja densidad de notación, no mala calidad. Se conserva como `acceptable/suitable` y se analizará como subgrupo para que el fondo no domine PSNR/SSIM.

Marca un checkbox únicamente cuando necesites ampliar una posible anomalía material: desenfoque claro, contraste insuficiente, inclinación apreciable o imposibilidad de procesar la página.

In [ ]:
review.batch_widget()

### 3b. Ampliar solo las anomalías

Pulsa **Actualizar cola**: el selector mostrará exclusivamente los IDs que excluiste al tramitar los lotes, no las 67 páginas originales. Selecciona un ID, pulsa **Mostrar página**, describe la anomalía y guarda. Al guardarla desaparecerá de la cola.

Las incidencias de calidad son casillas independientes que pueden activarse y desactivarse normalmente. La duplicación no se decide aquí.

In [ ]:
review.item_widget()

### 4. Revisar los 14 candidatos a duplicado

No necesitas recordar comparaciones anteriores. Al mostrar un par, el notebook enseña debajo de las imágenes todas las decisiones —revisadas o pendientes— que compartan cualquiera de esas dos páginas.

Compara el par actual y selecciona:

- `distinct`: páginas independientes;
- `related`: misma fuente o variantes, sin ser duplicados;
- `duplicate`: el mismo contenido visual, aunque cambien compresión, recorte o resolución;
- `unavailable`: la evidencia no permite decidir.

La etiqueta describe **el par que aparece en pantalla**. Si una página participa en varios pares y las relaciones son realmente diferentes, detente e informa a Codex: el notebook detectará el conflicto en vez de forzar una decisión incorrecta.

In [ ]:
review.candidate_widget()

## Checks

### 5. Comprobar el progreso

Esta comprobación es local y no finaliza el manifiesto. La validación estricta se ejecutará después desde GSD.

In [ ]:
review.progress_widget()

## Next Steps

Cuando aparezca **699/699 filas revisadas**, guarda el notebook y responde a Codex:

> revisión SMB lista

Codex ejecutará `validate-review` en modo de solo lectura, finalizará el manifiesto inmutable y repetirá todas las pruebas. No ejecutes `prepare-review`: sobrescribiría el CSV revisado.